Comenzamos cargando el dataset para este ejercicio: usaremos el dataset disponible en el repositorio de GitHub: **50-startups**

Este dataset también está disponible en la plataforma Kaggle.
Link: [Dataset](https://www.kaggle.com/farhanmd29/50-startups)

In [6]:
url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/50_Startups.csv"

Cargamos el dataset en un DataFrame de Pandas, y echamos un vistazo a las primeras 5 instancias.

In [7]:
import pandas as pd

df = pd.read_csv(url)
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


La variable objetivo a predecir (etiqueta) es **Profit** (rentabilidad de la Startup) y existen cuatro atributos predictores: tres de ellos numéricos y el cuarto es categórico (**State**). En este ejemplo omitiremos la información contenida en el atributo categórico, limitándonos únicamente a usar los tres numéricos.

In [8]:
df_num = df.drop('State', axis=1) # Eliminar atributo categórico
df_num.head()

,R&D Spend,Administration,Marketing Spend,Profit
0,165349.20,136897.80,471784.10,192261.83
1,162597.70,151377.59,443898.53,191792.06
2,153441.51,101145.55,407934.54,191050.39
3,144372.41,118671.85,383199.62,182901.99
4,142107.34,91391.77,366168.42,166187.94


Separar atributos predictores (X) de la etiqueta (y), y particionar el dataset en conjuntos de **entrenamiento** (80%) y **test** (20%).

El método *train_test_split()* de scikit-learn nos ayuda a hacer este particionado de forma aleatoria, mediante muestreo. Usar el argumento *random_state* es opcional, pero ayuda a que nuestro código sea reproducible.

In [9]:
X = df_num.drop('Profit',axis=1)
y = df_num['Profit']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

Entrenaremos un modelo de regresión lineal múltiple usando la clase using **LinearRegression** de *scikit-learn*, tras lo cual realizaremos predicciones sobre el conjunto de ejemplos de test.

In [10]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train,y_train) # Se usan tanto los atributos X como las etiquetas y para entrenar el modelo

y_predictions =  lin_reg.predict(X_test)
y_predictions

array([126703.02716461,  84894.75081556,  98893.41815974,  46501.70815036,
       129128.39734381,  50992.69486261, 109016.5536578 , 100878.4641454 ,
        97700.59638629, 113106.15292226])

Calcular el **error** (RMSE, root mean squared error) cometido en las predicciones hechas por el modelo:

In [11]:
from sklearn.metrics import mean_squared_error
import numpy as np

# CUIDADO: la raíz cuadrada no la aplica la función que usaremos. Tendremos que calcularla manualmente
lin_mse = mean_squared_error(y_test, y_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

np.float64(8995.905803361415)

**ENTRENANDO UN MODELO DE REGRESIÓN RIDGE Y LASSO**

In [12]:
from sklearn.linear_model import Ridge, Lasso

In [13]:
ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)

print("Coeficientes modelo de regresión lineal múltiple inicial: ", lin_reg.coef_)
print("Coeficientes modelo de regresión con regularización Ridge: ", ridge.coef_)

Coeficientes modelo de regresión lineal múltiple inicial:  [ 0.80377928 -0.06792917  0.03124155]
Coeficientes modelo de regresión con regularización Ridge:  [ 0.80377928 -0.06792917  0.03124155]


In [14]:
lasso = Lasso(alpha=1000.0)
lasso.fit(x_train, y_train)

print("Coeficientes modelo de regresión lineal múltiple inicial: ", lin_reg.coef_)
print("Coeficientes modelo de regresión con regularización Lasso: ", lasso.coef_)

Coeficientes modelo de regresión lineal múltiple inicial:  [ 0.80377928 -0.06792917  0.03124155]
Coeficientes modelo de regresión con regularización Lasso:  [ 0.80378043 -0.06792851  0.03124113]


In [15]:
y_predictions_ridge =  ridge.predict(x_test)

ridge_mse = mean_squared_error(y_test, y_predictions_ridge)
ridge_rmse = np.sqrt(ridge_mse)
ridge_rmse

np.float64(8995.905803402364)

In [16]:
y_predictions_lasso =  lasso.predict(x_test)

lasso_mse = mean_squared_error(y_test, y_predictions_lasso)
lasso_rmse = np.sqrt(lasso_mse)
lasso_rmse

np.float64(8995.879247147404)

La mejora es mínima, casi nula. ¿Por qué? Porque el modelo original no estaba sobreajustado, y los modelos entrenados con un mecanismo de regularización como Lasso (L1) y Ridge (L2) son efectivos cuando a priori existiera sobreajuste.

Veamos qué pasa si intencionalmente manipulamos el dataset para que dé lugar a un sobreajuste, por ejemplo, introduciendo ruido resultante de crear variables polinomiales:

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

# 1. Carga de datos
url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/50_Startups.csv"
df = pd.read_csv(url)

# 2. Separación de variables
X = df.drop(['State', 'Profit'], axis=1)
y = df['Profit']

# 3. Particionado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================================================================
# AÑADIENDO COMPLEJIDAD Y DIMENSIONALIDAD AL DATASET "ARTIFICIALMENTE"
# ==============================================================================

# A. Aumentamos la complejidad artificialmente (Grado 3 genera unas 20 variables)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# B. Escalado (ALTAMENTE RECOMENDABLE para Ridge y Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

# ==============================================================================
# ENTRENAMIENTO Y COMPARACIÓN DE MODELOS SIN REGULARIZAR Y REGULARIZADOS
# ==============================================================================

# 1. Regresión Lineal (Va a sobreajustar terriblemente)
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
lin_rmse = np.sqrt(mean_squared_error(y_test, lin_reg.predict(X_test_scaled)))

# 2. Ridge (Suavizará los coeficientes sin eliminarlos)
ridge = Ridge(alpha=10.0) # Un alpha más fuerte para domar el polinomio
ridge.fit(X_train_scaled, y_train)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_scaled)))

# 3. Lasso (Eliminará las variables polinomiales inútiles)
lasso = Lasso(alpha=500.0, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_scaled)))

print("--- RESULTADOS (RMSE) ---")
print(f"Regresión Lineal Pura:  {lin_rmse:,.2f}")
print(f"Regresión Ridge:        {ridge_rmse:,.2f}")
print(f"Regresión Lasso:        {lasso_rmse:,.2f}")

print("\n--- ANÁLISIS DE COEFICIENTES ---")
print(f"Total variables: {X_train_scaled.shape[1]}")
print(f"Variables anuladas (coef=0) por Lineal: {sum(lin_reg.coef_ == 0)}")
print(f"Variables anuladas (coef=0) por Ridge:  {sum(ridge.coef_ == 0)}")
print(f"Variables anuladas (coef=0) por Lasso:  {sum(lasso.coef_ == 0)}")

--- RESULTADOS (RMSE) ---
Regresión Lineal Pura:  25,104.17
Regresión Ridge:        12,879.69
Regresión Lasso:        8,599.70

--- ANÁLISIS DE COEFICIENTES ---
Total variables: 19
Variables anuladas (coef=0) por Lineal: 0
Variables anuladas (coef=0) por Ridge:  0
Variables anuladas (coef=0) por Lasso:  16
